# Transform Drivers Data

1. Read bronze `drivers` table
2. Keep only the columns required for Analytics (drop url column)
3. Standardise column names using snake_case
4. Concatenete `name.givenName` and `name.familyName` to create a new column called `driver_name` and transform values to Title Case
5. Filter out rows where `driver_id` is null
6. Remove duplicated values
8. Write the transformated data to a silver table 

In [0]:
%run ../00-common/01.enviroment-config

In [0]:
bronze_table = f'{catalog_name}.{bronze_schema}.drivers'
silver_table = f'{catalog_name}.{silver_schema}.drivers'

## Step 1 - Read bronze `drivers` table


In [0]:
drivers_df = spark.read.table(bronze_table)

In [0]:
display(drivers_df)

## Step 2 - Keep only the columns required for Analytics (drop url column)

In [0]:
drivers_dropped_df = drivers_df.drop('url')

##Step 3 - Standardise column names using snake_case


In [0]:
drivers_df_renamed = drivers_dropped_df.withColumnsRenamed(
    {
        'driverId': 'driver_id',
        'dateOfBirth': 'date_of_birth'
    }
)

##Step 4 - Concatenete `name.givenName` and `name.familyName` to create a new column called `driver_name` and transform values to Title Case

In [0]:
from pyspark.sql import functions as F

In [0]:
drivers_concateneted = (
    drivers_df_renamed.withColumn(
        'driver_name', 
        F.concat_ws(
            ' ', 
            drivers_df_renamed.name.givenName, 
            drivers_df_renamed.name.familyName
        )
    )
)

In [0]:
drivers_df_titled_case = (
    drivers_concateneted
        .withColumn(
            'driver_name', F.initcap('driver_name')
        )
        .withColumn(
            'nationality', F.initcap('nationality')
        )
        .drop('name'))

## Step 5 - Filter out rows where `driver_id` is null

In [0]:
drivers_df_not_null = drivers_df_titled_case.filter(F.col('driver_id').isNotNull())
drivers_df_not_null.count()

## Step 6 - Remove duplicated values

In [0]:
drivers_df_final = drivers_df_not_null.dropDuplicates(['driver_id'])
drivers_df_final.count()

## Step 7 - Write the transformted data to a silver table 

In [0]:
(
    drivers_df_final.write
        .format('delta')
        .mode('overwrite')
        .saveAsTable(silver_table)
)

In [0]:
%sql
SELECT * FROM formula1.silver.drivers where nationality is null